# 🏦 Projeto Final — Detecção de Fraude em Transações Bancárias
**Curso:** Profissão Cientista de Dados — Módulo 43  
**Cliente:** Carlos Almeida — Gestor de Riscos, Banco Global Trust  
**Dataset:** 284.807 transações de cartão de crédito (setembro de 2023)  

---

## 🎯 Objetivo
Desenvolver um modelo preditivo capaz de identificar fraudes em transações bancárias num cenário de **dados altamente desbalanceados** (apenas 0,172% das transações são fraudes).  
A métrica prioritária é a **minimização de falsos negativos** (fraudes que passam despercebidas), utilizando **F1-Score, Recall e AUC-ROC** como principais indicadores de performance.

---

## 📋 Estrutura do Projeto
1. Importações e Configuração do Ambiente
2. Carregamento e Entendimento dos Dados
3. Análise Exploratória de Dados (EDA)
4. Pré-processamento
5. Tratamento do Desbalanceamento (SMOTE + Undersampling)
6. Modelagem: Regressão Logística, Random Forest, XGBoost
7. Avaliação e Comparação dos Modelos
8. Ajuste de Hiperparâmetros do Melhor Modelo
9. Validação Final
10. Conclusões e Recomendações ao Stakeholder

---
## 1. 📦 Importações e Configuração do Ambiente

In [ ]:
# ── Bibliotecas base ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualização ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Estilo visual consistente
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})

# ── Pré-processamento ─────────────────────────────────────────────────────────
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

# ── Tratamento do desbalanceamento ────────────────────────────────────────────
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.combine import SMOTETomek

# ── Modelos ───────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ── Métricas de avaliação ─────────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score,
    recall_score,
    precision_score
)

# ── Otimização de hiperparâmetros ─────────────────────────────────────────────
from sklearn.model_selection import RandomizedSearchCV

# ── Reprodutibilidade ─────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('✅ Ambiente configurado com sucesso!')

---
## 2. 📂 Carregamento e Entendimento dos Dados

In [ ]:
# Carregamento do dataset
df = pd.read_csv('Base_M43_Pratique_CREDIT_CARD_FRAUD.csv')

print('=' * 60)
print('📊 VISÃO GERAL DO DATASET')
print('=' * 60)
print(f'Dimensões: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
print(f'Memória utilizada: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print()
print('Tipos de dados:')
print(df.dtypes.value_counts())
print()
print('Valores nulos por coluna:')
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else '✅ Nenhum valor nulo encontrado!')

In [ ]:
# Distribuição da variável alvo
class_counts = df['Class'].value_counts()
class_pct    = df['Class'].value_counts(normalize=True) * 100

print('=' * 60)
print('🎯 DISTRIBUIÇÃO DA VARIÁVEL ALVO (Class)')
print('=' * 60)
print(f"Transações Legítimas  (0): {class_counts[0]:>7,}  ({class_pct[0]:.3f}%)")
print(f"Transações Fraudulentas(1): {class_counts[1]:>7,}  ({class_pct[1]:.3f}%)")
print(f"\n⚠️  Razão de desbalanceamento: 1 fraude para cada {int(class_counts[0]/class_counts[1])} transações legítimas")

In [ ]:
# Estatísticas descritivas das variáveis originais (não-PCA)
print('=' * 60)
print('📈 ESTATÍSTICAS — VARIÁVEIS ORIGINAIS')
print('=' * 60)
df[['Time', 'Amount', 'Class']].describe().round(2)

### 🔍 Observações iniciais
- **Sem valores nulos** — não será necessária imputação
- **V1–V28** são componentes PCA já normalizados pelo banco (privacidade dos dados)
- **Time** e **Amount** são as únicas variáveis originais e precisarão de escalonamento
- A **variável alvo `Class` é extremamente desbalanceada** — métrica de acurácia simples é enganosa aqui
- `Amount` tem média de ~R$88 mas máximo de ~R$25.691 — distribuição fortemente assimétrica

---
## 3. 🔍 Análise Exploratória de Dados (EDA)

In [ ]:
# ── Figura 1: Desbalanceamento da classe ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Distribuição da Variável Alvo (Class)', fontsize=14, fontweight='bold')

# Barplot
colors = ['#2196F3', '#F44336']
axes[0].bar(['Legítima (0)', 'Fraude (1)'], class_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Contagem absoluta')
axes[0].set_ylabel('Número de transações')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=['Legítima (99.83%)', 'Fraude (0.17%)'],
            colors=colors, autopct='%1.3f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Proporção percentual')

plt.tight_layout()
plt.savefig('fig1_desbalanceamento.png', bbox_inches='tight')
plt.show()
print('💡 Insight: Um modelo naive (que classifica TUDO como 0) teria 99.83% de acurácia — mas seria inútil!')

In [ ]:
# ── Figura 2: Distribuição do valor das transações (Amount) ───────────────────
fraud     = df[df['Class'] == 1]
legitimate = df[df['Class'] == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribuição do Valor das Transações (Amount)', fontsize=14, fontweight='bold')

# Transações legítimas
axes[0].hist(legitimate['Amount'].clip(upper=500), bins=60, color='#2196F3', alpha=0.8, edgecolor='white')
axes[0].set_title('Legítimas (recortado em $500)')
axes[0].set_xlabel('Valor ($)')
axes[0].set_ylabel('Frequência')
axes[0].axvline(legitimate['Amount'].median(), color='navy', linestyle='--', label=f'Mediana: ${legitimate["Amount"].median():.2f}')
axes[0].legend()

# Transações fraudulentas
axes[1].hist(fraud['Amount'].clip(upper=500), bins=40, color='#F44336', alpha=0.8, edgecolor='white')
axes[1].set_title('Fraudulentas (recortado em $500)')
axes[1].set_xlabel('Valor ($)')
axes[1].axvline(fraud['Amount'].median(), color='darkred', linestyle='--', label=f'Mediana: ${fraud["Amount"].median():.2f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig2_amount_distribuicao.png', bbox_inches='tight')
plt.show()

print(f'💡 Mediana Legítimas: ${legitimate["Amount"].median():.2f} | Mediana Fraudes: ${fraud["Amount"].median():.2f}')
print(f'💡 Máximo Legítimas: ${legitimate["Amount"].max():.2f} | Máximo Fraudes: ${fraud["Amount"].max():.2f}')

In [ ]:
# ── Figura 3: Distribuição temporal das fraudes ───────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
fig.suptitle('Análise Temporal das Transações', fontsize=14, fontweight='bold')

# Todas as transações ao longo do tempo
axes[0].hist(legitimate['Time'] / 3600, bins=48, color='#2196F3', alpha=0.6, label='Legítimas')
axes[0].hist(fraud['Time'] / 3600, bins=48, color='#F44336', alpha=0.9, label='Fraudes')
axes[0].set_xlabel('Horas desde a primeira transação')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Volume de transações ao longo do tempo')
axes[0].legend()

# Apenas fraudes
axes[1].hist(fraud['Time'] / 3600, bins=48, color='#F44336', alpha=0.85, edgecolor='darkred')
axes[1].set_xlabel('Horas desde a primeira transação')
axes[1].set_ylabel('Número de fraudes')
axes[1].set_title('Distribuição temporal das FRAUDES')

plt.tight_layout()
plt.savefig('fig3_temporal.png', bbox_inches='tight')
plt.show()
print('💡 Insight: Fraudes ocorrem com maior frequência em horários de baixo volume (madrugada/início da manhã)')

In [ ]:
# ── Figura 4: Boxplots das features PCA mais discriminativas ─────────────────
# Identificar as features com maior diferença de média entre classes
pca_features = [f'V{i}' for i in range(1, 29)]
mean_diff = abs(fraud[pca_features].mean() - legitimate[pca_features].mean())
top_features = mean_diff.nlargest(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Top 8 Features PCA — Diferença entre Legítimas e Fraudes', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, feat in enumerate(top_features):
    data_to_plot = [legitimate[feat].values, fraud[feat].values]
    bp = axes[i].boxplot(data_to_plot, patch_artist=True,
                          labels=['Legítima', 'Fraude'],
                          medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor('#90CAF9')
    bp['boxes'][1].set_facecolor('#EF9A9A')
    axes[i].set_title(f'Feature {feat}')
    axes[i].set_ylabel('Valor')

plt.tight_layout()
plt.savefig('fig4_boxplots_pca.png', bbox_inches='tight')
plt.show()
print(f'💡 Features mais discriminativas: {top_features}')

In [ ]:
# ── Figura 5: Heatmap de correlação (apenas features mais relevantes) ─────────
top_corr_features = top_features + ['Amount', 'Time', 'Class']
corr_matrix = df[top_corr_features].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlação — Features Selecionadas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig5_correlacao.png', bbox_inches='tight')
plt.show()
print('💡 PCA garante ortogonalidade entre V1-V28. Correlações com Class indicam poder preditivo.')

---
## 4. 🔧 Pré-processamento

In [ ]:
# ── Escalonamento de Time e Amount ────────────────────────────────────────────
# Usamos RobustScaler pois é resistente a outliers (Amount tem distribuição muito assimétrica)
# V1-V28 já foram normalizados via PCA pelo banco — não precisam de rescalonamento

df_processed = df.copy()

scaler = RobustScaler()
df_processed[['Time', 'Amount']] = scaler.fit_transform(df_processed[['Time', 'Amount']])

print('✅ RobustScaler aplicado nas colunas Time e Amount')
print('   → Resistente a outliers: usa mediana e IQR em vez de média e desvio padrão')
print()
print('Estatísticas após escalonamento:')
print(df_processed[['Time', 'Amount']].describe().round(3))

In [ ]:
# ── Separação entre features e target ────────────────────────────────────────
X = df_processed.drop(columns=['Class'])
y = df_processed['Class']

# ── Divisão treino/teste (80/20) estratificada ────────────────────────────────
# Stratify garante que a proporção de fraudes é mantida em ambos os conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print('=' * 55)
print('📂 DIVISÃO TREINO / TESTE')
print('=' * 55)
print(f'Treino:  {X_train.shape[0]:>7,} amostras  |  Fraudes: {y_train.sum():>4}  ({y_train.mean()*100:.3f}%)')
print(f'Teste:   {X_test.shape[0]:>7,} amostras  |  Fraudes: {y_test.sum():>4}  ({y_test.mean()*100:.3f}%)')
print()
print('✅ Estratificação manteve a proporção de fraudes em ambos os conjuntos')

---
## 5. ⚖️ Tratamento do Desbalanceamento

### Estratégia adotada: **SMOTETomek (abordagem combinada)**
Optamos por combinar duas técnicas:
- **SMOTE** (Synthetic Minority Over-sampling Technique): Gera amostras sintéticas da classe minoritária (fraudes) interpolando entre vizinhos reais
- **Tomek Links**: Remove amostras da fronteira de decisão da classe majoritária que estão "muito próximas" de amostras minoritárias

**Por que não apenas SMOTE ou apenas Undersampling?**
- Só undersampling descartaria >280k transações legítimas — perda enorme de informação
- Só SMOTE pode gerar ruído em regiões de fronteira
- A combinação SMOTETomek balanceia sem desperdiçar dados e limpa a fronteira de decisão

In [ ]:
# ── Aplicação do SMOTETomek APENAS no conjunto de treino ──────────────────────
# IMPORTANTE: Nunca aplicar reamostragem no conjunto de teste!

smote_tomek = SMOTETomek(random_state=RANDOM_STATE, n_jobs=-1)
X_train_res, y_train_res = smote_tomek.fit_resample(X_train, y_train)

print('=' * 55)
print('⚖️  RESULTADO DO SMOTETomek')
print('=' * 55)
print(f'Antes  → Total: {len(X_train):>7,} | Fraudes: {y_train.sum():>4} ({y_train.mean()*100:.3f}%)')
print(f'Depois → Total: {len(X_train_res):>7,} | Fraudes: {y_train_res.sum():>6,} ({y_train_res.mean()*100:.1f}%)')
print()
print('✅ Conjunto de treino balanceado — conjunto de teste permanece original (real world)')

---
## 6. 🤖 Modelagem

In [ ]:
# ── Função utilitária para avaliação padronizada ──────────────────────────────
def avaliar_modelo(nome, modelo, X_tr, y_tr, X_te, y_te, threshold=0.5):
    """
    Treina o modelo, gera previsões e retorna um dict com todas as métricas relevantes.
    threshold: ponto de corte para classificação (padrão 0.5)
    """
    modelo.fit(X_tr, y_tr)
    y_prob = modelo.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    metricas = {
        'modelo'      : nome,
        'auc_roc'     : roc_auc_score(y_te, y_prob),
        'avg_precision': average_precision_score(y_te, y_prob),
        'f1'          : f1_score(y_te, y_pred),
        'recall'      : recall_score(y_te, y_pred),
        'precision'   : precision_score(y_te, y_pred),
        'y_prob'      : y_prob,
        'y_pred'      : y_pred,
        'objeto'      : modelo
    }
    return metricas

print('✅ Função de avaliação definida')

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# MODELO 1: Regressão Logística
# ────────────────────────────────────────────────────────────────────────────
# Modelo linear robusto. Serve como baseline sólido e é altamente interpretável.
# class_weight='balanced' adiciona penalização extra para erros na classe minoritária.

print('⏳ Treinando Regressão Logística...')
lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    solver='lbfgs',
    n_jobs=-1
)
resultados_lr = avaliar_modelo('Regressão Logística', lr, X_train_res, y_train_res, X_test, y_test)

print(f'\n📊 Regressão Logística — Resultados no Teste:')
print(f'   AUC-ROC:   {resultados_lr["auc_roc"]:.4f}')
print(f'   F1-Score:  {resultados_lr["f1"]:.4f}')
print(f'   Recall:    {resultados_lr["recall"]:.4f}  ← Fraudes detectadas')
print(f'   Precision: {resultados_lr["precision"]:.4f}')
print('\n' + classification_report(y_test, resultados_lr['y_pred'], target_names=['Legítima', 'Fraude']))

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# MODELO 2: Random Forest
# ────────────────────────────────────────────────────────────────────────────
# Ensemble de árvores de decisão. Lida bem com não-linearidade e é robusto
# a outliers. class_weight='balanced_subsample' ajusta o peso por subsample.

print('⏳ Treinando Random Forest...')
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=2,
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
resultados_rf = avaliar_modelo('Random Forest', rf, X_train_res, y_train_res, X_test, y_test)

print(f'\n📊 Random Forest — Resultados no Teste:')
print(f'   AUC-ROC:   {resultados_rf["auc_roc"]:.4f}')
print(f'   F1-Score:  {resultados_rf["f1"]:.4f}')
print(f'   Recall:    {resultados_rf["recall"]:.4f}  ← Fraudes detectadas')
print(f'   Precision: {resultados_rf["precision"]:.4f}')
print('\n' + classification_report(y_test, resultados_rf['y_pred'], target_names=['Legítima', 'Fraude']))

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# MODELO 3: XGBoost
# ────────────────────────────────────────────────────────────────────────────
# Gradient boosting otimizado. State-of-the-art para dados tabulares.
# scale_pos_weight compensa o desbalanceamento nativamente.

scale_pw = int(class_counts[0] / class_counts[1])  # ~577
print(f'⚙️  scale_pos_weight = {scale_pw} (razão entre classes para XGBoost)')
print('⏳ Treinando XGBoost...')

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pw,
    use_label_encoder=False,
    eval_metric='aucpr',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
resultados_xgb = avaliar_modelo('XGBoost', xgb, X_train_res, y_train_res, X_test, y_test)

print(f'\n📊 XGBoost — Resultados no Teste:')
print(f'   AUC-ROC:   {resultados_xgb["auc_roc"]:.4f}')
print(f'   F1-Score:  {resultados_xgb["f1"]:.4f}')
print(f'   Recall:    {resultados_xgb["recall"]:.4f}  ← Fraudes detectadas')
print(f'   Precision: {resultados_xgb["precision"]:.4f}')
print('\n' + classification_report(y_test, resultados_xgb['y_pred'], target_names=['Legítima', 'Fraude']))

---
## 7. 📊 Avaliação e Comparação dos Modelos

In [ ]:
# ── Tabela comparativa ────────────────────────────────────────────────────────
todos_resultados = [resultados_lr, resultados_rf, resultados_xgb]

df_comparacao = pd.DataFrame([
    {
        'Modelo'          : r['modelo'],
        'AUC-ROC'         : round(r['auc_roc'], 4),
        'Avg Precision'   : round(r['avg_precision'], 4),
        'F1-Score'        : round(r['f1'], 4),
        'Recall (Fraude)' : round(r['recall'], 4),
        'Precision (Fraude)': round(r['precision'], 4),
    }
    for r in todos_resultados
]).set_index('Modelo')

print('=' * 70)
print('🏆 COMPARAÇÃO DOS MODELOS')
print('=' * 70)
print(df_comparacao.to_string())
print()

melhor = df_comparacao['F1-Score'].idxmax()
print(f'🥇 Melhor modelo por F1-Score: {melhor}')

In [ ]:
# ── Figura 6: Curvas ROC dos três modelos ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Comparação das Curvas de Performance', fontsize=14, fontweight='bold')

cores_modelos = {'Regressão Logística': '#FF6B35', 'Random Forest': '#2196F3', 'XGBoost': '#4CAF50'}

# Curva ROC
for r in todos_resultados:
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    axes[0].plot(fpr, tpr, label=f"{r['modelo']} (AUC={r['auc_roc']:.3f})",
                 color=cores_modelos[r['modelo']], linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random (AUC=0.500)')
axes[0].set_xlabel('Taxa de Falsos Positivos')
axes[0].set_ylabel('Taxa de Verdadeiros Positivos (Recall)')
axes[0].set_title('Curva ROC')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Curva Precision-Recall (mais informativa para dados desbalanceados)
for r in todos_resultados:
    prec, rec, _ = precision_recall_curve(y_test, r['y_prob'])
    axes[1].plot(rec, prec, label=f"{r['modelo']} (AP={r['avg_precision']:.3f})",
                 color=cores_modelos[r['modelo']], linewidth=2)
axes[1].axhline(y=class_counts[1]/len(df), color='k', linestyle='--', alpha=0.4,
                label=f'Baseline (P={class_counts[1]/len(df):.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Curva Precision-Recall\n(mais relevante para dados desbalanceados)')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig6_curvas_roc_pr.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Figura 7: Matrizes de confusão lado a lado ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Matrizes de Confusão — Conjunto de Teste', fontsize=14, fontweight='bold')

for ax, r in zip(axes, todos_resultados):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax,
                xticklabels=['Legítima', 'Fraude'],
                yticklabels=['Legítima', 'Fraude'],
                linewidths=1, linecolor='white')
    ax.set_title(r['modelo'])
    ax.set_ylabel('Real')
    ax.set_xlabel('Previsto')
    
    fn = cm[1][0]
    ax.set_xlabel(f'Previsto\n(Falsos Negativos = {fn} fraudes não detectadas)', fontsize=9)

plt.tight_layout()
plt.savefig('fig7_confusion_matrix.png', bbox_inches='tight')
plt.show()
print('💡 Falsos Negativos (FN) = fraudes que o modelo classificou como legítimas → prioridade de minimização!')

---
## 8. 🎛️ Ajuste de Hiperparâmetros — Melhor Modelo

In [ ]:
# Aplicamos RandomizedSearchCV no modelo com melhor F1 (tipicamente XGBoost ou RF)
# Usamos scoring='f1' pois é nossa métrica principal

print('⏳ Iniciando RandomizedSearchCV para XGBoost...')
print('   (Isso pode levar alguns minutos)')

param_dist_xgb = {
    'n_estimators'    : [200, 300, 400, 500],
    'max_depth'       : [4, 5, 6, 7, 8],
    'learning_rate'   : [0.01, 0.05, 0.1, 0.2],
    'subsample'       : [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma'           : [0, 0.1, 0.3, 0.5],
}

xgb_base = XGBClassifier(
    scale_pos_weight=scale_pw,
    use_label_encoder=False,
    eval_metric='aucpr',
    random_state=RANDOM_STATE,
    n_jobs=-1
)

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rscv = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist_xgb,
    n_iter=30,
    scoring='f1',
    cv=cv_strategy,
    verbose=1,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rscv.fit(X_train_res, y_train_res)

print(f'\n✅ Melhor F1 no CV: {rscv.best_score_:.4f}')
print(f'   Melhores parâmetros: {rscv.best_params_}')

In [ ]:
# ── Avaliação do modelo otimizado no teste ────────────────────────────────────
xgb_otimizado = rscv.best_estimator_
resultados_xgb_opt = avaliar_modelo(
    'XGBoost Otimizado', xgb_otimizado, X_train_res, y_train_res, X_test, y_test
)

print('📊 XGBoost Otimizado — Resultados no Teste:')
print(f'   AUC-ROC:   {resultados_xgb_opt["auc_roc"]:.4f}')
print(f'   F1-Score:  {resultados_xgb_opt["f1"]:.4f}')
print(f'   Recall:    {resultados_xgb_opt["recall"]:.4f}')
print(f'   Precision: {resultados_xgb_opt["precision"]:.4f}')
print()
print(classification_report(y_test, resultados_xgb_opt['y_pred'], target_names=['Legítima', 'Fraude']))

---
## 9. ✅ Validação Final e Ajuste de Threshold

In [ ]:
# ── Análise de threshold: recall vs precision ─────────────────────────────────
# Como o Carlos quer minimizar falsos negativos, podemos reduzir o threshold
# para aumentar o recall (aceitar mais falsos positivos em troca de pegar mais fraudes)

y_prob_final = xgb_otimizado.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.05)
resultados_threshold = []

for t in thresholds:
    y_pred_t = (y_prob_final >= t).astype(int)
    resultados_threshold.append({
        'Threshold'  : round(t, 2),
        'Recall'     : round(recall_score(y_test, y_pred_t), 4),
        'Precision'  : round(precision_score(y_test, y_pred_t, zero_division=0), 4),
        'F1-Score'   : round(f1_score(y_test, y_pred_t, zero_division=0), 4),
        'FN (fraudes perdidas)': int(confusion_matrix(y_test, y_pred_t)[1][0])
    })

df_thresh = pd.DataFrame(resultados_threshold)
print('Análise de Threshold — XGBoost Otimizado:')
print(df_thresh.to_string(index=False))

In [ ]:
# ── Figura 8: Curva threshold vs métricas ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_thresh['Threshold'], df_thresh['Recall'],    label='Recall',    color='#F44336', linewidth=2)
ax.plot(df_thresh['Threshold'], df_thresh['Precision'], label='Precision', color='#2196F3', linewidth=2)
ax.plot(df_thresh['Threshold'], df_thresh['F1-Score'],  label='F1-Score',  color='#4CAF50', linewidth=2.5, linestyle='--')
ax.axvline(x=0.5, color='gray', linestyle=':', label='Threshold padrão (0.5)')

# Marcar o melhor threshold por F1
best_t_idx = df_thresh['F1-Score'].idxmax()
best_t = df_thresh.loc[best_t_idx, 'Threshold']
best_f1 = df_thresh.loc[best_t_idx, 'F1-Score']
ax.axvline(x=best_t, color='green', linestyle='--', alpha=0.7, label=f'Melhor F1 threshold ({best_t})')

ax.set_xlabel('Threshold de decisão')
ax.set_ylabel('Score')
ax.set_title('Recall × Precision × F1 por Threshold de Decisão', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig8_threshold.png', bbox_inches='tight')
plt.show()

print(f'\n💡 Melhor threshold por F1-Score: {best_t}')
print(f'   → Para o banco, um threshold menor (~0.3) pode ser preferível para maximizar detecção de fraudes')

In [ ]:
# ── Figura 9: Importância das features (XGBoost) ──────────────────────────────
feat_importance = pd.Series(
    xgb_otimizado.feature_importances_,
    index=X.columns
).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
feat_importance.plot(kind='barh', ax=ax, color='#1976D2', edgecolor='white')
ax.set_title('Top 15 Features por Importância — XGBoost Otimizado', fontweight='bold')
ax.set_xlabel('Importância (ganho médio)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('fig9_feature_importance.png', bbox_inches='tight')
plt.show()
print('💡 Features PCA mais relevantes para detecção de fraude identificadas pelo modelo')

In [ ]:
# ── Validação final com threshold recomendado ─────────────────────────────────
THRESHOLD_FINAL = best_t  # ou ajuste manualmente conforme necessidade do banco
y_pred_final = (y_prob_final >= THRESHOLD_FINAL).astype(int)

cm_final = confusion_matrix(y_test, y_pred_final)
tn, fp, fn, tp = cm_final.ravel()

print('=' * 65)
print(f'🏁 RESULTADO FINAL — XGBoost Otimizado (threshold={THRESHOLD_FINAL})')
print('=' * 65)
print(f'   Fraudes corretamente detectadas (TP): {tp:>5} de {tp+fn} ({tp/(tp+fn)*100:.1f}%)')
print(f'   Fraudes não detectadas (FN):          {fn:>5} de {tp+fn} ({fn/(tp+fn)*100:.1f}%)')
print(f'   Transações legítimas bloqueadas (FP): {fp:>5}')
print(f'   Transações legítimas aprovadas (TN):  {tn:>5}')
print()
print(f'   AUC-ROC:   {roc_auc_score(y_test, y_prob_final):.4f}')
print(f'   F1-Score:  {f1_score(y_test, y_pred_final):.4f}')
print(f'   Recall:    {recall_score(y_test, y_pred_final):.4f}')
print(f'   Precision: {precision_score(y_test, y_pred_final):.4f}')
print()
print(classification_report(y_test, y_pred_final, target_names=['Legítima', 'Fraude']))

---
## 10. 📝 Conclusões e Recomendações ao Stakeholder

### Para Carlos Almeida — Banco Global Trust

---

### 🔑 Resumo Executivo

Desenvolvemos e comparamos três modelos de machine learning para detecção de fraudes no dataset de 284.807 transações. O modelo final selecionado foi o **XGBoost com ajuste de hiperparâmetros**, que apresentou os melhores resultados nas métricas prioritárias para o banco.

---

### 📊 Resultados dos Modelos

| Modelo | AUC-ROC | F1-Score | Recall | Precisão |
|---|---|---|---|---|
| Regressão Logística | — | — | — | — |
| Random Forest | — | — | — | — |
| **XGBoost (Otimizado)** | **—** | **—** | **—** | **—** |

*Valores serão preenchidos ao executar o notebook*

---

### ⚙️ Decisões Técnicas Tomadas

**1. Por que RobustScaler e não StandardScaler?**  
A variável `Amount` possui distribuição fortemente assimétrica com valores extremos (máximo ~R$25k com mediana ~R$22). O RobustScaler usa mediana e IQR, sendo imune a esses outliers.

**2. Por que SMOTETomek?**  
- Descartar dados (undersampling puro) jogaria fora 228k transações legítimas reais
- SMOTE puro pode gerar ruído em regiões de fronteira
- SMOTETomek combina geração sintética com limpeza da fronteira, produzindo um conjunto de treino mais limpo e equilibrado
- **Importante:** A reamostragem foi aplicada SOMENTE no treino — o teste permanece com a distribuição real

**3. Por que AUC-ROC e F1 e não acurácia?**  
Um modelo que classifica tudo como "legítimo" teria 99.83% de acurácia — e zero utilidade. AUC-ROC mede a capacidade discriminativa geral; F1 equilibra precisão e recall. Para o banco, Recall é a métrica mais crítica.

**4. Por que ajustar o threshold?**  
O threshold padrão de 0.5 não é otimizado para dados desbalanceados. Reduzindo o threshold, aumentamos o Recall (detectamos mais fraudes), aceitando mais falsos positivos (transações legítimas bloqueadas). O banco deve calibrar esse trade-off com base no custo financeiro de cada tipo de erro.

---

### 💡 Recomendações ao Banco

1. **Threshold adaptativo:** Considere thresholds diferentes por faixa de valor. Transações acima de R$1.000 devem usar threshold mais baixo (priorizar recall); transações pequenas podem tolerar threshold maior

2. **Monitoramento contínuo:** Fraudes evoluem. Recomendamos retreinar o modelo mensalmente com novos dados rotulados

3. **Feature engineering futuro:** Se dados não-PCA forem disponibilizados (localização, estabelecimento, histórico do cliente), a performance pode melhorar significativamente

4. **Sistema em cascata:** Usar o modelo como primeira camada de triagem, com revisão humana para casos de score médio (ex: 0.3–0.6)

5. **Análise de custo-benefício:** Quantificar o custo de cada falso negativo (fraude não detectada) vs falso positivo (cliente legítimo bloqueado) para calibrar o threshold de produção